# LMCache on CPU: with and without

Two copies of vLLM run the same small model (`Qwen/Qwen2.5-0.5B-Instruct`) on this machine's CPU. One is plain; the other hands its KV cache to an LMCache server that keeps it in RAM and on disk. A 6,000-token document about Samsung is loaded into LMCache once. Then you ask questions about it, both copies answer side by side, and you watch the time to first token.

Launch this notebook from the repo's `.venv` on Python 3.12 (see README). Run the cells top to bottom; every command is in the open.

## 1. vLLM's CPU build

Published on PyPI as `vllm-cpu-nightly`, with wheels for Python 3.12 only. The `apache-tvm-ffi` cap is what LMCache's CI applies: newer versions crash xgrammar at import on macOS.

In [ ]:
%pip install vllm-cpu-nightly "apache-tvm-ffi<0.1.13" --extra-index-url https://download.pytorch.org/whl/cpu

The wheel installs under the name `vllm-cpu-nightly`, but `vllm serve` looks its version up under the name `vllm`, and the CPU platform plugin keys off a `+cpu` version label. Register that name:

In [ ]:
import importlib.metadata as md, pathlib, shutil
d = md.distribution("vllm-cpu-nightly")
info = next(p.parts[0] for p in d.files if p.parts[0].endswith(".dist-info"))
src = pathlib.Path(d.locate_file("")) / info
dst = src.with_name(f"vllm-{d.version}+cpu.dist-info")
shutil.rmtree(dst, ignore_errors=True)
shutil.copytree(src, dst)
meta = dst / "METADATA"
meta.write_text(meta.read_text()
    .replace("Name: vllm-cpu-nightly", "Name: vllm", 1)
    .replace(f"Version: {d.version}", f"Version: {d.version}+cpu", 1))
print(dst.name)

In [ ]:
!python -c "import vllm, torch; print('vllm ok!', vllm.__version__, 'cuda:', torch.cuda.is_available())"

## 2. LMCache

From the `dev` branch, without GPU extensions, and with `--no-deps` so the CPU torch installed above stays in place. pip prints a red opentelemetry version warning on the requirements step; ignore it.

In [ ]:
!git clone --depth 1 --branch dev https://github.com/LMCache/LMCache.git

In [ ]:
%pip install -r LMCache/requirements/build.txt -r LMCache/requirements/common.txt -r LMCache/requirements/cli.txt

In [ ]:
%env NO_GPU_EXT=1
%env SETUPTOOLS_SCM_PRETEND_VERSION=0.0.0

In [ ]:
%pip install --no-deps --no-build-isolation -e LMCache

In [ ]:
!python -c "import lmcache; print('lmcache ok!')"

## 3. The UI library

In [ ]:
%pip install gradio ipywidgets

## 4. Start the LMCache server

2 GB of KV cache in RAM (L1), and every entry also written to disk under `./l2` (L2). It listens for vLLM on `localhost:5555` and serves metrics on `localhost:8080`.

In [ ]:
# stop anything left from an earlier run, then confirm its ports are free ([r] keeps pkill from matching this line)
!pkill -f "lmcache serve[r]"; sleep 2; lsof -i :5555 -i :8080 2>/dev/null | grep -v COMMAND || echo "ports 5555 and 8080 are free"

In [ ]:
%%bash --bg
lmcache server \
    --l1-size-gb 2 --eviction-policy LRU \
    --supported-transfer-mode engine_driven --shm-name lmc \
    --l2-adapter '{"type":"fs","base_path":"l2"}' \
    > lmcache.log 2>&1

In [ ]:
!until curl -sf localhost:8080/healthcheck > /dev/null; do pgrep -f "lmcache serve[r]" > /dev/null || { echo "lmcache server exited. Last lines of lmcache.log:"; tail -n 12 lmcache.log | sed "s/\x1b\[[0-9;]*m//g"; break; }; grep "LMCache" lmcache.log | tail -n 1 | sed "s/\x1b\[[0-9;]*m//g" | cut -c 1-160; sleep 1; done; curl -sf localhost:8080/healthcheck > /dev/null && echo "lmcache server up"

Turn on the store controller's debug lines, so writes to disk show up in `lmcache.log`:

In [ ]:
!curl -s "localhost:8080/loglevel?logger_name=lmcache.v1.distributed.storage_controllers.store_controller&level=DEBUG"

## 5. Start two copies of vLLM

Both download `Qwen/Qwen2.5-0.5B-Instruct` from Hugging Face on first start (about 1 GB). Both have vLLM's own prefix cache off: the baseline so that it really recomputes the document each time, the LMCache one so that every hit you see is LMCache's. The only difference is the `--kv-transfer-config` on the second.

In [ ]:
!pkill -f "vllm serv[e]"; sleep 2; lsof -i :8000 -i :8001 2>/dev/null | grep -v COMMAND || echo "ports 8000 and 8001 are free"

In [ ]:
%%bash --bg
vllm serve Qwen/Qwen2.5-0.5B-Instruct --port 8001 \
    --max-model-len 8192 --kv-cache-memory-bytes 2147483648 --enforce-eager \
    --no-enable-prefix-caching \
    > vllm_baseline.log 2>&1

In [ ]:
%%bash --bg
vllm serve Qwen/Qwen2.5-0.5B-Instruct --port 8000 \
    --max-model-len 8192 --kv-cache-memory-bytes 2147483648 --enforce-eager \
    --no-enable-prefix-caching \
    --kv-transfer-config '{"kv_connector":"LMCacheMPConnector","kv_role":"kv_both","kv_connector_module_path":"lmcache.integration.vllm.lmcache_mp_connector"}' \
    > vllm_lmcache.log 2>&1

In [ ]:
!until curl -s localhost:8001/v1/models | grep -q Qwen2.5; do pgrep -f "vllm serve .* --port 800[1]" > /dev/null || { echo "baseline vllm exited. Last lines of vllm_baseline.log:"; tail -n 12 vllm_baseline.log; break; }; tail -n 1 vllm_baseline.log | sed "s/\x1b\[[0-9;]*m//g" | cut -c 1-160; sleep 5; done; curl -s localhost:8001/v1/models | grep -q Qwen2.5 && echo "baseline vllm up (:8001)"

In [ ]:
!until curl -s localhost:8000/v1/models | grep -q Qwen2.5; do pgrep -f "vllm serve .* --port 800[0]" > /dev/null || { echo "vllm + LMCache exited. Last lines of vllm_lmcache.log:"; tail -n 12 vllm_lmcache.log; break; }; tail -n 1 vllm_lmcache.log | sed "s/\x1b\[[0-9;]*m//g" | cut -c 1-160; sleep 5; done; curl -s localhost:8000/v1/models | grep -q Qwen2.5 && echo "vllm + LMCache up (:8000)"

## 6. The document

`samsung.txt` is the Wikipedia article on Samsung. It becomes the system prompt of every question, so its KV cache is one shared prefix.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
doc = open("samsung.txt").read()
doc = tok.decode(tok(doc).input_ids[:6000])          # cap the context at 6,000 tokens
print(len(tok(doc).input_ids), "tokens of context\n")
print(doc[:700], "...")

## 7. Client code

The three servers, the messages (document first, question last), `send_request()` for one plain request, a reader for LMCache's counters and log, and the streaming code the UI uses to run both servers at once.

In [ ]:
import requests, json, time, threading, queue, os, re

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
VLLM = "http://localhost:8001"            # plain vLLM
VLLM_LMCACHE = "http://localhost:8000"    # vLLM with the LMCache connector
LMCACHE = "http://localhost:8080"         # the LMCache server itself (metrics, status)

def build_messages(question):
    # The document goes first, identical for every question, so its KV cache is a shared prefix.
    return [{"role": "system", "content": "Answer the question using this document.\n\n" + doc},
            {"role": "user", "content": question}]

def send_request(server, question, max_tokens=120):
    # One chat completion. Raises on any HTTP error; returns the server's JSON reply.
    r = requests.post(f"{server}/v1/chat/completions", timeout=900,
                      json={"model": MODEL, "messages": build_messages(question), "max_tokens": max_tokens, "temperature": 0})
    r.raise_for_status()
    return r.json()

def get_lmcache_counter(name):
    # Current value of one LMCache server counter, e.g. "l1_write_chunks_total".
    for line in requests.get(f"{LMCACHE}/metrics").text.splitlines():
        if line.split("{")[0] == f"lmcache_mp_{name}":
            return float(line.split()[-1])
    return 0.0

def get_lmcache_log_since(offset):
    # LMCache server log lines written since `offset` (a byte position in lmcache.log).
    with open("lmcache.log") as f:
        f.seek(offset)
        lines = [re.sub(r"\x1b\[[0-9;]*m", "", l) for l in f]
    lines = [re.sub(r" \([a-z_]+\.py:\d+:[^)]*\)\s*$", "", l) for l in lines]      # drop the "(file.py:line:module)" suffix
    return "\n".join(l for l in lines if "Retrieved" in l or "Stored" in l or "Prefetch request completed" in l)

def get_lmcache_hits():
    # Tokens LMCache has served from L1 so far.
    return get_lmcache_counter("lookup_hit_l1_tokens_total")

# ---- streaming, used by the UI: both servers answer the same question at the same time ----

SIDES = {"vLLM": VLLM, "vLLM + LMCache": VLLM_LMCACHE}

def stream_request(side, question, q):
    # Stream one answer from one server into the queue as (side, time_to_first_token, text_chunk).
    body = {"model": MODEL, "messages": build_messages(question), "max_tokens": 120, "temperature": 0, "stream": True}
    t0, ttft = time.time(), None
    with requests.post(f"{SIDES[side]}/v1/chat/completions", json=body, stream=True, timeout=900) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            if not line.startswith(b"data: ") or line == b"data: [DONE]":
                continue
            delta = json.loads(line[6:])["choices"][0]["delta"].get("content") or ""
            if delta:
                ttft = ttft or time.time() - t0
                q.put((side, ttft, delta))
    q.put((side, ttft, None))

def ask_both_servers(question):
    # Yields the two answer boxes as they fill in, then the LMCache log lines for the request.
    q = queue.Queue()
    text = {s: "" for s in SIDES}; ttft = {s: None for s in SIDES}; done = 0
    hits_before = get_lmcache_hits()
    log_offset = os.path.getsize("lmcache.log")
    for side in SIDES:
        threading.Thread(target=stream_request, args=(side, question, q), daemon=True).start()
    label = lambda s: f"**time to first token: {ttft[s]:.2f} s**" if ttft[s] else "waiting for first token …"
    while done < len(SIDES):
        side, t, delta = q.get()
        ttft[side] = t
        if delta is None:
            done += 1
        else:
            text[side] += delta
        yield label("vLLM"), text["vLLM"], label("vLLM + LMCache"), text["vLLM + LMCache"], ""
    time.sleep(0.5)
    served = get_lmcache_hits() - hits_before
    files = [os.path.join(d, f) for d, _, fs in os.walk("l2") for f in fs if f.endswith(".data")]
    disk = f"{len(files)} KV files on disk, {sum(os.path.getsize(f) for f in files) / 1e6:.0f} MB"
    yield label("vLLM"), text["vLLM"], label("vLLM + LMCache") + f" · {served:.0f} context tokens served by LMCache · {disk}", text["vLLM + LMCache"], get_lmcache_log_since(log_offset)

## 8. The UI

Press **Ask both** twice with the same question. The first time both sides compute the document and the LMCache log shows `Stored … tokens`; the second time the right side is served from LMCache and the log shows `Retrieved … tokens`. Then change the question: the document is the shared prefix, so the right side keeps hitting. The URL printed under the cell works in a browser tab for projecting.

In [ ]:
import gradio as gr
gr.close_all()          # so re-running this cell frees port 7860 first

def tail_logs(n=200):
    # Last n lines of each server log, colour codes stripped, metrics/healthcheck polling lines dropped.
    def tail(path):
        try:
            lines = open(path).read().splitlines()
        except FileNotFoundError:
            lines = []
        lines = [l for l in lines if '"GET /metrics' not in l and '"GET /healthcheck' not in l][-n:]
        return "\n".join(re.sub(r"\x1b\[[0-9;]*m", "", l)[:200] for l in lines)
    return tail("lmcache.log"), tail("vllm_baseline.log"), tail("vllm_lmcache.log")

with gr.Blocks(title="LMCache on CPU") as ui:
    gr.Markdown("## Same model, same CPU, same question. Left is plain vLLM; right is vLLM with LMCache.\n"
                "Question 1: neither side has the document's KV cache, both compute it, LMCache stores it. "
                "Question 2 onward: the left recomputes it every time, the right gets it back from LMCache.")
    with gr.Accordion(f"The document ({len(tok(doc).input_ids):,} tokens, sent with every question)", open=False):
        gr.Textbox(value=doc, lines=12, show_label=False)
    question = gr.Textbox(label="Ask about the document", value="When was Samsung founded, and by whom?")
    gr.Examples(["What businesses is Samsung involved in?",
                 "Which Samsung affiliates are listed on the stock exchange?",
                 "What happened to Samsung in the 1990s?",
                 "Who leads Samsung today?"], inputs=question)
    go = gr.Button("Ask both", variant="primary")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### vLLM")
            ttft_base = gr.Markdown("")
            out_base = gr.Textbox(label="answer", lines=8)
        with gr.Column():
            gr.Markdown("### vLLM + LMCache")
            ttft_lmc = gr.Markdown("")
            out_lmc = gr.Textbox(label="answer", lines=8)
            log_lmc = gr.Textbox(label="LMCache server log for this request", lines=5)
    outputs = [ttft_base, out_base, ttft_lmc, out_lmc, log_lmc]
    go.click(ask_both_servers, inputs=question, outputs=outputs)
    question.submit(ask_both_servers, inputs=question, outputs=outputs)
    gr.Markdown("### Live logs")
    with gr.Row():
        tail_lmcache = gr.Textbox(label="lmcache.log", lines=12, max_lines=12, autoscroll=True)
        tail_vllm = gr.Textbox(label="vllm_baseline.log", lines=12, max_lines=12, autoscroll=True)
        tail_vllm_lmcache = gr.Textbox(label="vllm_lmcache.log", lines=12, max_lines=12, autoscroll=True)
    gr.Timer(1.0).tick(tail_logs, outputs=[tail_lmcache, tail_vllm, tail_vllm_lmcache])

ui.launch(inline=True, prevent_thread_lock=True)   # first free port from 7860 up; the URL is printed below

## 9. Stop

In [ ]:
# [e] keeps pkill from matching this very command line
!pkill -f "vllm serv[e]"; pkill -f "lmcache serve[r]"; sleep 3; echo stopped

---
Logs: `lmcache.log`, `vllm_baseline.log`, `vllm_lmcache.log` (also tailed live at the bottom of the UI). Server state: http://localhost:8080/status · metrics: http://localhost:8080/metrics